# ship
> E2E deployment: provision VPS, generate Dockerfile + Compose stack, deploy app, backup/restore

In [ ]:
#| default_exp ship

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os, subprocess, tempfile, time
from tempfile import NamedTemporaryFile as ntf
import ujson as json
import yaml
from datetime import datetime
from fastcore.all import L, Path, parse_env, last, dict2obj, joins, bind, not_, equals, AttrDictDefault, true
from fastops.core import Compose, env_set, env_get, detect_app, secret_get, Dockerfile
from fastops.proxy import caddy_svc, cloudflared_svc, crowdsec as crowdsec_svc
from fastops.vps import (vps_init, create, delete, deploy, deploy_mp, run_ssh, wait_ssh,
                         chk_docker, multi_init, Multipass, load_pub_keys, sync)
from fastops.cloudflare import CF

## Env helpers

`read_env_keys()` parses `.env.example` for key names — returns `[]` if the file is absent so callers never need to guard. `check_envs()` warns about keys that are missing or empty in the provided dict. `write_remote_env()` pushes a merged env dict to the server as `/srv/app/.env` via scp.

In [ ]:
#| export
def read_env_keys(dir, fn='.env.example') -> list:
    'Parse .env.example key names from src dir. Returns [] if file absent.'
    return parse_env(fn=p).keys() if (p := Path(dir/fn)) and p.exists() else []

def check_envs(keys: list, provided: dict) -> list:
    'Warn about keys missing/empty in provided. Returns list of missing keys.'
    return L(keys).filter(not_(provided.get))

def write_remote_env(host, envs: dict, path='/srv/app/.env', user='deploy', key=None):
    'Write env dict to .env file on remote host via scp.'
    con = joins('\n', L(envs).map(lambda kv: f'{kv[0]}={kv[1]}'))
    with ntf(mode='w', suffix='.env') as f:
        f.write(con)
        run_ssh(host, f'mkdir -p {str(Path(path).parent)}', user=user, key=key)
        args = ['scp', '-o', 'StrictHostKeyChecking=accept-new'] + (['-i', str(key)] if key else [])
        subprocess.run(args + [f.name, f'{user}@{host}:{path}'], check=True)

In [ ]:
env=parse_env('FOO=val\nBAR=other\n# comment\nBAZ=\n')
check_envs(['FOO', 'BAR', 'BAZ', 'QUX'], env)

['BAZ', 'QUX']

In [ ]:
#| export
def env_push(name_or_host=None, envs=None, env_file=None, srv_path='/srv/app', user='deploy', key=None):
    'Merge new env vars into remote .env — adds/updates keys, preserves unmentioned ones.'
    h, p = _resolve(name_or_host or env_get('FASTOPS_VPS_HOST', default=''), srv_path)
    if not h: raise ValueError('host required')
    if not (merged := _merge_envs(env_file, envs)): return
    try: curr = parse_env(run_ssh(h, f'cat {p}/.env', user=user, key=key, capture=True))
    except: curr = {}
    write_remote_env(h, curr | merged, path=f'{p}/.env', user=user, key=key)

In [ ]:
# read_env_keys: present file
tf = tempfile
d = Path(tf.mkdtemp())
(d /'.env.example').write_text('# comment\nFOO=\nBAR=val\n\nBAZ=\n')
print(read_env_keys(d))
assert equals(read_env_keys(d), ['FOO', 'BAR', 'BAZ'])
# read_env_keys: missing file returns []
assert read_env_keys(Path(tf.mkdtemp())) == []
# check_envs
missing = check_envs(['FOO', 'BAR', 'BAZ'], {'FOO': 'x', 'BAZ': 'z'})
assert missing == ['BAR']
print('read_env_keys + check_envs OK')

dict_keys(['FOO', 'BAR', 'BAZ'])
read_env_keys + check_envs OK


## Stack builder

`stack()` builds a `docker-compose.yml` + `Caddyfile` for a given topology. It replaces the old `_build_stack` / `_compose_stack` pair with a single composable function.

`app_volumes`: dict `{'./data': '/app/data', ...}` — added as bind mounts on the `app` service. Use for persistent data dirs and large assets that are not baked into the Docker image.

`_port()`: reads the `EXPOSE` port from a `Dockerfile` object or string — used to feed `stack()` automatically from `detect_app()` output.

## Env management

`parse_env(fn=path)` (fastcore) parses a `.env` file to a dict — handles comments, blank lines, quoted values, and inline `# comments`. `check_envs()` warns about keys in `.env.example` not present in the provided dict (non-blocking). `write_remote_env()` pushes a dict as a `.env` file to a remote host via `scp`.

Two patterns for passing secrets, combinable — `envs` dict overrides `env_file` for the same key:

```python
# File (commit non-secrets, gitignore secrets)
launch('./myapp', env_file='./prod.env')

# Dict (CI / keyring)
launch('./myapp', envs={'JWT_SCRT': secret_get('JWT_SCRT')})

# Both
launch('./myapp', env_file='./prod.env', envs={'JWT_SCRT': secret_get('JWT_SCRT')})
```

In [ ]:
#| export
def _port(df) -> int:
    '''Return the last EXPOSE port in a Dockerfile object or string.
    Uses the last occurrence so multistage builds return the runtime stage port.
    Returns 8000 if no EXPOSE found.'''
    sw = lambda l: l.startswith('EXPOSE') and len(l.split()) > 1
    cvt = lambda l: int(l.split()[1])
    return last(L.splitlines(str(df)).filter(sw).map(cvt)) or 8000

def stack(domain, port, src=None, *, tunnel=True, crowdsec=False, email=None, dns=None,
          app_volumes=None, secure_env=True) -> Compose:
    '''Build Compose stack: app + caddy + optional cloudflared/crowdsec.
    secure_env=True: mount .env as read-only bind-mount (/app/.env) instead of injecting
    into process env — secrets invisible to docker inspect.
    app_volumes: {local_rel_path: container_path} additional bind mounts on the app service.
    If src given, writes docker-compose.yml + Caddyfile to that directory.'''
    cf_path = str(src/'Caddyfile') if src else 'Caddyfile'
    env_vol = {'./.env': '/app/.env:ro'} if secure_env else {}
    all_vols = {**env_vol, **(app_volumes or {})} or None
    app_svc = dict(build='.', networks=['web'], restart='unless-stopped', volumes=all_vols)
    caddy = caddy_svc(domain, port=port, cloudflared=tunnel, crowdsec=crowdsec, email=email, dns=dns, conf=cf_path)
    dc = Compose().svc('app', **app_svc).svc('caddy', **caddy)
    if tunnel: dc = dc.svc('cloudflared', **cloudflared_svc())
    if crowdsec: dc = dc.svc('crowdsec', **crowdsec_svc())
    dc = dc.network('web').volume('caddy_data').volume('caddy_config')
    if crowdsec: dc = dc.volume('crowdsec-db').volume('crowdsec-config')
    if src: dc.save(str(src/'docker-compose.yml'))
    return dc

In [ ]:
from fastops.core import fasthtml_app, python_app, go_app

In [ ]:
tf2 = tempfile
# _port: single EXPOSE
assert _port(fasthtml_app()) == 5001
assert _port(python_app(port=9000)) == 9000
assert _port('FROM alpine\nEXPOSE 3000\nCMD sh') == 3000
assert _port('FROM alpine') == 8000   # fallback
print('_port() single EXPOSE OK')

# _port: multistage — returns LAST EXPOSE (runtime stage), not builder stage
multistage = 'FROM golang AS builder\nEXPOSE 9999\nFROM alpine\nEXPOSE 8080\nCMD ["/app"]'
assert _port(multistage) == 8080
assert _port(go_app(port=8080)) == 8080
print('_port() multistage OK')

# stack: basic tunnel topology
with tf2.TemporaryDirectory() as tmp:
    src = Path(tmp)
    dc = stack('myapp.example.com', 5001, src, tunnel=True)
    d = dc.to_dict()
    assert 'app' in d['services'] and 'caddy' in d['services'] and 'cloudflared' in d['services']
    assert 'ports' not in d['services']['caddy']   # no open ports with tunnel
    assert (src/'docker-compose.yml').exists() and (src/'Caddyfile').exists()
    print('stack() tunnel OK')

# stack: direct TLS (tunnel=False)
with tf2.TemporaryDirectory() as tmp:
    dc = stack('myapp.example.com', 8000, Path(tmp), tunnel=False)
    d = dc.to_dict()
    assert 'cloudflared' not in d['services']
    assert '80:80' in d['services']['caddy']['ports']
    print('stack() direct TLS OK')

# stack: app_volumes added to app service
with tf2.TemporaryDirectory() as tmp:
    dc = stack('myapp.example.com', 5001, Path(tmp), tunnel=True,
               app_volumes={'./data': '/app/data', './static': '/app/static'})
    d = dc.to_dict()
    assert './data:/app/data' in d['services']['app']['volumes']
    assert './static:/app/static' in d['services']['app']['volumes']
    print('stack() app_volumes OK')

# stack: crowdsec
with tf2.TemporaryDirectory() as tmp:
    dc = stack('myapp.example.com', 5001, Path(tmp), tunnel=True, crowdsec=True)
    d = dc.to_dict()
    assert 'crowdsec' in d['services'] and 'crowdsec-db' in d['volumes']
    print('stack() crowdsec OK')

_port() single EXPOSE OK
_port() multistage OK
stack() tunnel OK
stack() direct TLS OK
stack() app_volumes OK
stack() crowdsec OK


## Deploy state

`record_deploy()` / `list_deploys()` / `get_deploy()` track what's been deployed using a JSON file at `~/.config/fastops/deploys.json`. This powers `backup()` and `restore()` — you can refer to an app by name instead of passing host/path every time.

In [ ]:
#| export
_deploys = Path.home() / '.config' / 'fastops' / 'deploys.json'
def _load_deploys(path:Path=_deploys) -> dict: return path.read_json() if path.exists() else {}
def _save_deploys(data: dict, path:Path=_deploys): path.write_json(data, indent=2)

def record_deploy(name, host, domain, src, srv_path='/srv/app', vm_name='', status='active', version='', path=_deploys):
    'Upsert a deploy record.'
    d = _load_deploys(path)
    d[name] = dict(name=name, host=host, domain=domain, src=str(src), srv_path=srv_path, vm_name=vm_name or '',
                status=status, version=version, deployed_at=datetime.now().isoformat())
    _save_deploys(d, path)

def list_deploys(path=_deploys) -> dict:
    'Return list of all deploy records.'
    return dict2obj(_load_deploys(path), dict_func=AttrDictDefault)

def get_deploy(name, path=_deploys)-> dict | None:
    'Return deploy record for name, or None.'
    return list_deploys(path).get(name)

In [ ]:
import tempfile as _tf3

_test_json = Path(_tf3.mktemp(suffix='.json'))

record_deploy('myapp', '1.2.3.4', 'myapp.example.com', '/code/myapp', path=_test_json)
record_deploy('myapp', '1.2.3.5', 'myapp.example.com', '/code/myapp', version='def456', path=_test_json)
dt = get_deploy('myapp', path=_test_json)
assert dt.host == '1.2.3.5'                      # updated
assert dt.version == 'def456'
assert dt.status == 'active'
record_deploy('otherapp', '2.3.4.5', 'other.example.com', '/code/other', status='stopped', path=_test_json)
assert get_deploy('otherapp', path=_test_json).status == 'stopped'
assert get_deploy('nope', path=_test_json) is None
print('deploy state JSON OK')
_test_json.unlink(missing_ok=True)

deploy state JSON OK


## Provision, wait, ship

`provision()` creates a Hetzner VPS with cloud-init and stores the IP in `FASTOPS_VPS_HOST`.

`wait_for_ready()` blocks until SSH is up and Docker is confirmed running — call it after `provision()` before the first `ship()`.

`ship()` rsyncs the app to the host and runs `docker compose up -d`.  
- `extra_dirs`: additional local subdirs to rsync separately (e.g. `['static', 'onnx-community']`). Synced into `srv_path/dirname/` alongside the app.  
- `exclude`: dirs to protect from `--delete` in the main rsync (e.g. `['data', 'backups']` — runtime-only, never in source).

In [ ]:
#| export
xclude = ['data', 'backups', '.venv', '__pycache__', '.git', '*.pyc']
ign = ['.git', '.venv', '__pycache__', '*.pyc', '.env', 'data', 'backups', 'node_modules', '*.tar.gz', '.DS_Store']

def mk_dock_ign(path, extra=None):
    'Write .dockerignore to path if absent. Prevents build context bloat.'
    p = Path(path) / '.dockerignore'
    if not p.exists(): p.write_text('\n'.join(ign + list(extra or [])) + '\n')

def provision(name, pub_keys, server_type='cx22', location='nbg1', image='ubuntu-24.04', docker=True, ssh_keys=None):
    'Create Hetzner VPS with cloud-init. Stores IP in FASTOPS_VPS_HOST env var. Returns IP.'
    ci = vps_init(name, pub_keys, docker=docker)
    raw = create(name, image=image, server_type=server_type, location=location, cloud_init=ci, ssh_keys=ssh_keys or [])
    ip = next(iter(raw)) if isinstance(raw, dict) else str(raw)
    env_set('FASTOPS_VPS_HOST', ip)
    env_set('FASTOPS_VPS_NAME', name)
    return ip

def is_ready(host, user='deploy', key=None, timeout=120):
    'Wait for SSH then verify Docker is available. Returns True or raises.'
    wait_ssh(host, u=user, k=key, tout=timeout)
    if not chk_docker(host, u=user, k=key): raise RuntimeError(f'Docker not ready on {host}')
    return True

def code_snap(host, path, name, user='deploy', key=None) -> str:
    'Tar current remote app dir to /tmp before overwriting. Returns remote tar path.'
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    remote_tar = f'/tmp/code-{name}-{ts}.tar.gz'
    try: run_ssh(host, f'test -d {path} && tar -czf {remote_tar} -C {path} .', user=user, key=key)
    except: return ''
    return remote_tar

def _deploy(path, host, user='deploy', key=None, srv_path='/srv/app', build=True,
            extra_dirs=None, exclude=None, snapshot=True):
    '''Rsync app to host and run docker compose up -d. Internal helper — use ship() for smart deploys.
    snapshot=True: snapshot remote code before overwriting (enables rollback()).
    build=True: pass --build to docker compose up to rebuild the image from source.'''
    if snapshot: code_snap(host, srv_path, Path(path).name, user, key)
    deploy(path, host, user=user, key=key, path=srv_path, build=build, exclude=list(exclude or xclude))
    fn = lambda p: sync(str(Path(path)/p).rstrip('/') + '/', f'{srv_path}/{p}', host, user=user, key=key)
    L(extra_dirs).filter(lambda p: (Path(path)/p).exists()).map(fn)
    return host

In [ ]:
#| export
def rollback(name=None, srv_path='/srv/app', user='deploy', key=None):
    'Restore the latest code snapshot and restart compose. Run after a bad deploy.'
    h, p = _resolve(name or env_get('FASTOPS_VPS_HOST', default=''), srv_path)
    if not h: raise ValueError('host required')
    nm = get_deploy(name).name or name or h
    tar = f'ls -t /tmp/code-{nm}-*.tar.gz 2>/dev/null | head -1'
    snap = run_ssh(h, tar, user=user, key=key, capture=True).strip()
    if not snap: raise FileNotFoundError(f'No code snapshot found for {nm!r} — was snapshot=True on last deploy?')
    run_ssh(h, f'tar -xzf {snap} -C {p}', user=user, key=key)
    run_ssh(h, f'cd {p} && docker compose up -d --remove-orphans --build', user=user, key=key)
    print(f'Rolled back {nm!r} from {snap}')

def health_check(name_or_host=None, srv_path='/srv/app', timeout=60, interval=5, user='deploy', key=None) -> bool:
    'Poll app container until healthy/running or timeout. Returns True or raises.'
    h, p = _resolve(name_or_host or env_get('FASTOPS_VPS_HOST', default=''), srv_path)
    if not h: raise ValueError('host required')
    dl = time.time() + timeout
    while time.time() < dl:
        try:
            cmd = f'docker compose -f {p}/docker-compose.yml ps --format json'
            out = run_ssh(h, cmd, user=user, key=key, capture=True).strip()
            for line in out.splitlines():
                if not line.strip(): continue
                svc = json.loads(line)
                if svc.get('Service') != 'app': continue
                st, health = svc.get('State', ''), svc.get('Health', '')
                if health == 'healthy': return True
                if st == 'running' and not health: return True  # no HEALTHCHECK defined — running is enough
                if st in ('exited', 'dead'): raise RuntimeError(f'App container {st} on {h}')
        except (json.JSONDecodeError, subprocess.CalledProcessError): pass
        time.sleep(interval)
    raise TimeoutError(f'App not healthy on {h} after {timeout}s')

## Backup and restore

`backup()` stops the app container, tars the data dirs and `.env` on the remote, downloads the archive, then restarts the app. Pass an app name (looked up from the deploy DB) or a `host` directly.

`restore()` uploads a backup archive, extracts it, and restarts the stack.

In [ ]:
#| export
_BACKUP_DIR = Path.home() / '.fastops' / 'backups'

def _resolve(name_or_host, srv_path) -> tuple[str, str]:
    'Return (host, srv_path) from a name (deploy record lookup) or literal host string.'
    rec = get_deploy(name_or_host)
    if rec: return rec.host, rec.srv_path
    return name_or_host, srv_path

def backup(name_or_host=None, data_dirs=('data',), srv_path='/srv/app', out=None, user='deploy', key=None) -> Path:
    '''Stop app → tar data_dirs + .env on remote → scp locally → restart app.
    name_or_host: app name (deploy record lookup) or IP. Defaults to FASTOPS_VPS_HOST.'''
    h, p = _resolve(name_or_host or env_get('FASTOPS_VPS_HOST', default=''), srv_path)
    if not h: raise ValueError('host required — pass name_or_host= or run provision() first.')
    nm, ts = name_or_host or h, datetime.now().strftime('%Y%m%d_%H%M%S')
    remote_tar = f'/tmp/backup-{nm}-{ts}.tar.gz'
    dirs = L(data_dirs).map(lambda d:f'{p}/{d}').map(bind(joins,sep=' '))
    tar_src = f'{dirs} {p}/.env'
    run_ssh(h, f'docker compose -f {p}/docker-compose.yml stop app', user=user, key=key)
    try:
        run_ssh(h, f'tar -czf {remote_tar} --ignore-failed-read {tar_src}', user=user, key=key)
        local = Path(out) if out else _BACKUP_DIR / f'{nm}-{ts}.tar.gz'
        local.parent.mkdir(parents=True, exist_ok=True)
        scp = ['scp', '-o', 'StrictHostKeyChecking=accept-new'] + (['-i', str(key)] if key else [])
        subprocess.run(scp + [f'{user}@{h}:{remote_tar}', str(local)], check=True)
        run_ssh(h, f'rm -f {remote_tar}', user=user, key=key)
    finally: run_ssh(h, f'docker compose -f {p}/docker-compose.yml start app', user=user, key=key)
    print(f'Backup saved: {local}')
    return local

def restore(backup_path, name_or_host=None, srv_path='/srv/app', user='deploy', key=None):
    'Upload backup archive, extract to srv_path, restart stack.'
    h, p = _resolve(name_or_host or env_get('FASTOPS_VPS_HOST', default=''), srv_path)
    if not h: raise ValueError('host required.')
    remote_tar = f'/tmp/restore-{Path(backup_path).name}'
    run_ssh(h, f'docker compose -f {p}/docker-compose.yml stop app', user=user, key=key)
    try:
        scp = ['scp', '-o', 'StrictHostKeyChecking=accept-new'] + (['-i', str(key)] if key else [])
        subprocess.run(scp + [str(backup_path), f'{user}@{h}:{remote_tar}'], check=True)
        run_ssh(h, f'tar -xzf {remote_tar} -C / && rm -f {remote_tar}', user=user, key=key)
    finally: run_ssh(h, f'docker compose -f {p}/docker-compose.yml start app', user=user, key=key)
    print(f'Restore complete from {backup_path}')

In [ ]:
#| export
def logs(name_or_host=None, service='app', n=100, follow=True, srv_path='/srv/app', user='deploy', key=None):
    '''Stream or tail docker compose logs. follow=True streams until Ctrl-C.
    Works for both VPS (SSH) and Multipass VMs (detected via deploy record).'''
    h, p = _resolve(name_or_host or env_get('FASTOPS_VPS_HOST', default=''), srv_path)
    if not h: raise ValueError('host required')
    rec = get_deploy(name_or_host) if isinstance(name_or_host, str) else None
    flags = ['--tail', str(n)] + (['-f'] if follow else [])
    if rec and getattr(rec, 'vm_name', ''):
        Multipass().exec_(rec.vm_name, 'docker', 'compose', '-f', f'{p}/docker-compose.yml', 'logs', *flags, service)
    else:
        f_str = f'--tail={n}' + (' -f' if follow else '')
        cmd = f'docker compose -f {p}/docker-compose.yml logs {f_str} {service}'
        run_ssh(h, cmd, user=user, key=key, check=False)  # check=False: Ctrl-C exits non-zero

def compose_cmd(cmd, name_or_host=None, service=None, srv_path='/srv/app', user='deploy', key=None):
    'Run a docker compose subcommand on the remote stack. cmd: restart, down, pull, etc.'
    h, p = _resolve(name_or_host or env_get('FASTOPS_VPS_HOST', default=''), srv_path)
    if not h: raise ValueError('host required')
    svc = f' {service}' if service else ''
    return run_ssh(h, f'docker compose -f {p}/docker-compose.yml {cmd}{svc}', user=user, key=key)

## ship / ship_local — smart entry points

`ship()` and `ship_local()` are the primary entry points for deployment. They manage state automatically:

- **First call** (no deploy record): runs the full provisioning flow — Dockerfile, CF tunnel, cloud-init, VPS/VM, wait, push env, rsync, `compose up --build`
- **Subsequent calls** (record exists): skips provisioning — just rsyncs and runs `compose up --build`

Pass `force=True` to re-provision from scratch even when a record exists.

`launch()` / `launch_local()` remain as explicit first-time-only provisioners (useful when you want to guarantee a fresh server regardless of state).

### Typical workflow

```python
src = Path('../orgs/myapp')
keys = read_env_keys(src)
envs = {k: secret_get(k) for k in keys}

# First call: provisions + deploys
ip = ship_local(src, 'myapp.example.com', envs=envs)

# Subsequent calls: just redeploys (build=False to skip image rebuild)
ship_local(src)

# Production
ip = ship(src, 'myapp.example.com', envs=envs, cf_token=...)
ship(src)  # redeploy
```

In [ ]:
#| export
def _clone_repo(url) -> Path:
    'Shallow-clone a git URL into a fresh tmpdir.'
    dst = Path(tempfile.mkdtemp())
    subprocess.run(['git', 'clone', '--depth', '1', url, str(dst)], check=True)
    return dst

def get_repo(repo):
    'Return a Path to the repo, cloning if repo is a git URL.'
    s = str(repo)
    return _clone_repo(s) if s.startswith(('http', 'git@')) else Path(repo)

def get_df(repo, **kw) -> tuple[Path, Dockerfile]:
    'Return (src_path, Dockerfile) for the repo, generating Dockerfile + .dockerignore if absent.'
    src = get_repo(repo)
    dfp = src / 'Dockerfile'
    df = detect_app(src, **kw).save(dfp) if not dfp.exists() else Dockerfile.load(dfp)
    mk_dock_ign(src)
    return src, df

def _merge_envs(env_file=None, envs=None) -> dict:
    'Merge env_file and envs dict. envs overrides env_file for the same key.'
    base = parse_env(fn=env_file) if (env_file and Path(env_file).exists()) else {}
    return base | (envs or {})

def _cf_setup(domain, name, ip=None, tunnel=True, token=None) -> str | None:
    '''Setup Cloudflare for a deploy. Returns CF_TUNNEL_TOKEN string if tunnel=True, else None.
    tunnel=True:  creates/reuses CF tunnel + CNAME → returns token for CF_TUNNEL_TOKEN.
    tunnel=False: adds a proxied DNS A record pointing ip at the domain (ip required).'''
    if not token: return None
    if tunnel: return CF(token).setup_tunnel(domain, name)[1]
    if ip: CF(token).upsert_record(domain, name, ip, type='A', proxied=True)
    return None

def launch(repo, domain, envs=None, env_file=None, pub_keys=None, ssh_keys=None, key=None, tunnel=True, cf_token=None,
    server_type='cx23', location='nbg1', image='ubuntu-24.04', crowdsec=False, app_volumes=None, extra_dirs=None,
    exclude=None, user='deploy', build=True, **kw) -> str:
    '''Provision a new Hetzner VPS and deploy repo. Returns IP.
    tunnel=True:  CF tunnel + CNAME (no open ports). Requires cf_token.
    tunnel=False: direct TLS via Caddy (ports 80/443).
    build=True: rebuild Docker image on deploy (default).
    **kw forwarded to detect_app().'''
    src, df = get_df(repo, **kw)
    merged = _merge_envs(env_file, envs)
    if tunnel and (tok:=_cf_setup(domain, src.name, tunnel=True, token=cf_token)): merged['CF_TUNNEL_TOKEN'] = tok
    stack(domain, _port(df), src, tunnel=bool(merged.get('CF_TUNNEL_TOKEN')), crowdsec=crowdsec, app_volumes=app_volumes)
    check_envs(read_env_keys(src), merged)
    ip = provision(src.name, load_pub_keys(pub_keys), server_type, location, image, ssh_keys=ssh_keys or [])
    is_ready(ip, user=user, key=key)
    if not tunnel: _cf_setup(domain, src.name, ip, tunnel=False, token=cf_token)
    if merged: write_remote_env(ip, merged, user=user, key=key)
    _deploy(src, ip, user=user, key=key, extra_dirs=extra_dirs, exclude=exclude, build=build, snapshot=False)
    record_deploy(src.name, ip, domain, src)
    env_set('FASTOPS_VPS_HOST', ip); env_set('FASTOPS_VPS_NAME', src.name)
    return ip

def launch_local(repo, domain, *, envs=None, env_file=None, pub_keys=None, cf_token=None, tunnel=True, crowdsec=False,
    image='24.04', cpus=1, memory='2G', disk='10G', app_volumes=None, extra_dirs=None, build=True, **kw) -> str:
    '''Provision a new Multipass VM and deploy repo. Returns VM IP.
    tunnel=True (default): CF tunnel dials outbound from inside the VM — no open ports needed.
    build=True: rebuild Docker image on deploy (default).
    **kw forwarded to detect_app().'''
    src, df = get_df(repo, **kw)
    merged = _merge_envs(env_file, envs)
    if tunnel and (tok:=_cf_setup(domain, src.name, tunnel=True, token=cf_token)): merged['CF_TUNNEL_TOKEN'] = tok
    stack(domain, _port(df), src, tunnel=bool(merged.get('CF_TUNNEL_TOKEN')), crowdsec=crowdsec, app_volumes=app_volumes)
    ci_yaml = multi_init(src.name, load_pub_keys(pub_keys))
    with ntf(mode='w', suffix='.yaml', delete=False) as f:
	    f.write(ci_yaml)
	    ci_path = f.name
    try: Multipass().launch(src.name, image=image, cpus=cpus, memory=memory, disk=disk, cloud_init=ci_path)
    finally: os.unlink(ci_path)
    vm_ip = Multipass().ip(src.name)
    wait_ssh(vm_ip, u='deploy', tout=180)
    if merged: write_remote_env(vm_ip, merged, user='deploy')
    deploy_mp(src.name, src, build=build)
    L(extra_dirs).filter(lambda p: (Path(src)/p).exists()).map(
        lambda p: Multipass().transfer(str(Path(src)/p).rstrip('/') + '/', f'{src.name}:/srv/app/{p}'))
    record_deploy(src.name, vm_ip, domain, src, vm_name=src.name)
    return vm_ip

In [ ]:
#| export
def ship(repo, domain=None, *, force=False, build=True, user='deploy', key=None,
         srv_path='/srv/app', extra_dirs=None, exclude=None,
         envs=None, env_file=None, pub_keys=None, ssh_keys=None, tunnel=True, cf_token=None,
         server_type='cx23', location='nbg1', image='ubuntu-24.04', crowdsec=False,
         app_volumes=None, setup_ci=True, gh_token=None, **kw) -> str:
    '''Smart deploy to Hetzner VPS. Provisions on first call, redeploys on subsequent calls. Returns IP.
    force=True: re-provision even if a deploy record exists.
    build=True: rebuild Docker image on every deploy (default). Pass False to skip rebuild.
    gh_token: GitHub token (or True to read GITHUB_TOKEN env var) — creates .fastops/config.json
      and pushes .github/workflows/fastops.yml via ghapi on first deploy.
    Provisioning kwargs (envs, cf_token, server_type, etc.) only used on first deploy.'''
    src = get_repo(repo)
    rec = get_deploy(src.name)
    if rec and not force:
        _deploy(src, rec.host, user, key=key, srv_path=srv_path, build=build, extra_dirs=extra_dirs, exclude=exclude)
        record_deploy(src.name, rec.host, rec.domain, src, srv_path=srv_path)
        return rec.host
    if not domain: raise ValueError('domain required for first deploy — not found in deploy records')
    ip = launch(src, domain, envs=envs, env_file=env_file, pub_keys=pub_keys, ssh_keys=ssh_keys,
                key=key, tunnel=tunnel, cf_token=cf_token, server_type=server_type, location=location,
                image=image, crowdsec=crowdsec, app_volumes=app_volumes, extra_dirs=extra_dirs,
                exclude=exclude, user=user, build=build, **kw)
    if setup_ci and (tok := _resolve_gh_token(gh_token)):
        fo_init(src.name, ip, domain, branch=_git_branch(src), path=src)
        fo_hook(tok, path=src)
    return ip

def ship_local(repo, domain=None, *, force=False, build=True,
               envs=None, env_file=None, pub_keys=None, cf_token=None, tunnel=True,
               crowdsec=False, image='24.04', cpus=1, memory='2G', disk='10G',
               app_volumes=None, extra_dirs=None, **kw) -> str:
    '''Smart local deploy via Multipass. Provisions VM on first call, redeploys on subsequent calls. Returns VM IP.
    force=True: re-provision even if a deploy record exists.
    build=True: rebuild Docker image on every deploy (default).'''
    src = get_repo(repo)
    rec = get_deploy(src.name)
    if rec and rec.vm_name and not force:
        deploy_mp(src.name, src, build=build)
        L(extra_dirs).filter(lambda p: (Path(src)/p).exists()).map(
            lambda p: Multipass().transfer(str(Path(src)/p).rstrip('/') + '/', f'{src.name}:/srv/app/{p}'))
        record_deploy(src.name, rec.host, rec.domain, src, vm_name=src.name)
        return rec.host
    if not domain: raise ValueError('domain required for first deploy — not found in deploy records')
    return launch_local(src, domain, envs=envs, env_file=env_file, pub_keys=pub_keys,
                        cf_token=cf_token, tunnel=tunnel, crowdsec=crowdsec, image=image,
                        cpus=cpus, memory=memory, disk=disk, app_volumes=app_volumes,
                        extra_dirs=extra_dirs, build=build, **kw)

In [ ]:
#| export
def teardown(name, cf_token=None, delete_server=True, user='deploy', key=None):
    'Remove server/VM, optionally delete CF tunnel, mark deploy destroyed.'
    rec = get_deploy(name)
    if not rec: raise ValueError(f'No deploy record for {name!r}')
    if getattr(rec, 'vm_name', ''):
        try: Multipass().rm(rec.vm_name)
        except Exception as e: print(f'VM removal failed: {e}')
    elif delete_server:
        try: delete(name)
        except Exception as e: print(f'Server deletion failed: {e}')
    if cf_token:
        try: CF(cf_token).delete_tunnel(name)
        except Exception as e: print(f'CF tunnel removal failed: {e}')
    record_deploy(name, rec.host, rec.domain, rec.src, status='destroyed')
    print(f'Teardown complete: {name!r}')

def teardown_local(name, cf_token=None):
    'Remove Multipass VM for a local deploy. Thin alias for teardown() — detects VM via deploy record.'
    teardown(name, cf_token=cf_token, delete_server=False)

## Safe deploy + GitHub workflow

`deploy_safe()` is the CI entry point: code snapshot → data backup → deploy → health check → auto-rollback on failure. It reads host config from `.fastops/config.json` when present, so it works without arguments in CI (the repo is checked out, config is there).

The `fo_*` functions manage `.fastops/config.json` — a git-tracked file that lives in the app repo. It records deploy environments (host, domain, branch) and CI settings (test command, health path). `fo_hook()` reads it and pushes a GitHub Actions workflow via ghapi.

Pass `gh_token=True` to `ship()` to automate `fo_init` + `fo_hook` on first deploy:

```python
ip = ship(repo, 'myapp.com', envs=envs, gh_token=True)
# → provisions VPS, writes .fastops/config.json, pushes .github/workflows/fastops.yml
# → prints: gh secret set VPS_SSH_KEY ...
```

Add environments later:

```python
fo_add_env('staging', host='5.6.7.8', domain='staging.myapp.com', branch='dev')
fo_hook(token=True)  # regenerates + pushes updated workflow
```

In [ ]:
#| export
def cfg_path(path='.') -> Path:
    'Return path to .fastops/config.json.'
    return Path(path) / '.fastops' / 'config.json'

def _read_fo_config(path='.') -> dict:
    'Read .fastops/config.json. Returns {} if absent.'
    return p.read_json() if (p := cfg_path(path)).exists() else {}

def _write_fo_config(cfg: dict, path='.'):
    'Write .fastops/config.json, creating dirs as needed.'
    p = cfg_path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_json(cfg, indent=2)

def _git_branch(path='.') -> str:
    'Return current git branch. Falls back to "main".'
    try:
        return subprocess.check_output(
            ['git', '-C', str(path), 'rev-parse', '--abbrev-ref', 'HEAD'],
            stderr=subprocess.DEVNULL, text=True).strip() or 'main'
    except Exception: return 'main'

def _detect_test(path='.') -> str | None:
    'Return "pytest" if pytest is in pyproject.toml dev deps, else None.'
    try:
        if 'pytest' in (Path(path) / 'pyproject.toml').read_text(): return 'pytest'
    except Exception: pass

def _get_repo_slug(path='.') -> tuple[str, str]:
    'Return (owner, repo) from git remote origin URL.'
    try:
        url = subprocess.check_output(
            ['git', '-C', str(path), 'remote', 'get-url', 'origin'],
            stderr=subprocess.DEVNULL, text=True).strip()
        parts = url.rstrip('/').removesuffix('.git').replace(':', '/').split('/')
        return parts[-2], parts[-1]
    except Exception: raise ValueError(f'Could not parse git remote origin in {path!r}')

def _resolve_gh_token(gh_token) -> str | None:
    'Tiered resolution: explicit str → GITHUB_TOKEN env → .fastops/token file → FASTSHIP_TOKEN env.'
    if isinstance(gh_token, str): return gh_token
    for k in ('GITHUB_TOKEN', 'FASTSHIP_TOKEN'):
        if v := os.environ.get(k): return v
    tok_file = Path('.fastops') / 'token'
    if tok_file.exists(): return tok_file.read_text().strip() or None
    return None

def _gh_api(token, path='.') -> tuple:
    'Return (owner, repo_name, GhApi) for the repo at path.'
    try: from ghapi.all import GhApi
    except ImportError: raise ImportError('ghapi required: uv add ghapi')
    owner, repo_name = _get_repo_slug(path)
    return owner, repo_name, GhApi(token=token)

def fo_record_deploy(env, sha, path='.'):
    'Write deploy state back to .fastops/config.json. Called by CI after successful deploy.'
    cfg = _read_fo_config(path)
    cfg.setdefault('deploys', {})[env] = dict(
        version=sha, deployed_at=datetime.now().isoformat(), status='active')
    _write_fo_config(cfg, path)

def _print_required_secrets(cfg):
    print('\nRequired GitHub secrets:')
    print('  gh secret set VPS_SSH_KEY < ~/.ssh/id_ed25519')
    for env_name, ec in cfg.get('envs', {}).items():
        s = f'VPS_HOST_{env_name.upper()}'
        print(f'  gh secret set {s} --body "{ec["host"]}"')

In [ ]:
#| export
def fo_init(name, host, domain, *, env='prod', branch=None, srv_path='/srv/app',
            health_path='/health', test_cmd=None, env_schema=None, path='.') -> Path:
    '''Create/update .fastops/config.json in the app repo. Returns config path.
    env_schema: {KEY: None} → GitHub Secret, {KEY: "default"} → GitHub Variable.
    Auto-detects branch and test command if not given.'''
    cfg = _read_fo_config(path)
    cfg.setdefault('app', name)
    cfg.setdefault('envs', {})[env] = dict(
        host=host, domain=domain, branch=branch or _git_branch(path), srv_path=srv_path)
    cfg.setdefault('ci', {})['health_path'] = health_path
    if test_cmd: cfg['ci']['test_cmd'] = test_cmd
    elif not cfg['ci'].get('test_cmd') and (t := _detect_test(path)): cfg['ci']['test_cmd'] = t
    if env_schema is not None: cfg['env_schema'] = env_schema
    _write_fo_config(cfg, path)
    print(f'Config: {cfg_path(path)}')
    return cfg_path(path)

def fo_add_env(env, host, domain, *, branch, srv_path='/srv/app', path='.'):
    '''Add/update an environment in .fastops/config.json and regenerate the workflow locally.
    Call fo_hook() afterwards to push the updated workflow to GitHub.'''
    cfg = _read_fo_config(path)
    if not cfg: raise ValueError('No .fastops/config.json — run fo_init() first')
    cfg.setdefault('envs', {})[env] = dict(host=host, domain=domain, branch=branch, srv_path=srv_path)
    _write_fo_config(cfg, path)
    wf = Path(path) / '.github' / 'workflows' / 'fastops.yml'
    wf.parent.mkdir(parents=True, exist_ok=True)
    wf.write_text(mk_workflow(cfg))
    print(f'Added env {env!r}. Review {wf} then run fo_hook() to push.')

def fo_status(path='.'):
    'Print deploy state for all environments in .fastops/config.json.'
    cfg = _read_fo_config(path)
    if not cfg: print('No .fastops/config.json'); return
    print(f'App: {cfg.get("app")}')
    for env, ec in cfg.get('envs', {}).items():
        d = cfg.get('deploys', {}).get(env, {})
        ver = (d.get('version') or 'never')[:8]
        ts  = (d.get('deployed_at') or '—')[:19]
        st  = d.get('status', '—')
        print(f'  {env:12} {ec["domain"]:30} {ver:10} {ts}  [{st}]')

def mk_workflow(cfg: dict) -> str:
    '''Generate GitHub Actions YAML from .fastops/config.json. Pure — no side effects.
    Builds one deploy job per environment, triggered on its configured branch.
    Includes a test job if ci.test_cmd is set.'''
    app, envs, ci = cfg.get('app', 'app'), cfg.get('envs', {}), cfg.get('ci', {})
    test_cmd = ci.get('test_cmd')
    branches = sorted(set(e.get('branch', 'main') for e in envs.values()))
    jobs = {}
    if test_cmd:
        jobs['test'] = {'runs-on': 'ubuntu-latest', 'steps': [
            {'uses': 'actions/checkout@v4'}, {'uses': 'astral-sh/setup-uv@v4'},
            {'run': f'uv run {test_cmd}'},
        ]}
    for env_name, ec in envs.items():
        branch = ec.get('branch', 'main')
        host_s = f'VPS_HOST_{env_name.upper()}'
        steps = [
            {'uses': 'actions/checkout@v4'}, {'uses': 'astral-sh/setup-uv@v4'},
            {'run': 'uv add fastops --dev'},
            {'name': 'Setup SSH', 'run': '\n'.join([
                'mkdir -p ~/.ssh',
                f'echo "${{{{ secrets.VPS_SSH_KEY }}}}" > ~/.ssh/deploy_key',
                'chmod 600 ~/.ssh/deploy_key',
                f'ssh-keyscan -H "${{{{ secrets.{host_s} }}}}" >> ~/.ssh/known_hosts',
            ])},
            {'name': 'Deploy', 'run': (
                "uv run python - << 'EOF'\n"
                f"from fastops import deploy_safe\nimport sys\n"
                f"sys.exit(0 if deploy_safe('{env_name}', key='$HOME/.ssh/deploy_key') else 1)\n"
                "EOF"
            )},
            {'name': 'Record version', 'if': 'success()', 'run': '\n'.join([
                'git config user.email "fastops-bot@noreply"',
                'git config user.name "fastops"',
                f"uv run python -c \"from fastops import fo_record_deploy; fo_record_deploy('{env_name}', '${{{{ github.sha }}}}')\"",
                'git add .fastops/config.json',
                f'git diff --cached --quiet || git commit -m "deploy: {env_name} @ ${{{{ github.sha }}}}"',
                'git push',
            ])},
        ]
        job = {'runs-on': 'ubuntu-latest',
               'if': f"github.ref == 'refs/heads/{branch}'",
               'permissions': {'contents': 'write'}, 'steps': steps}
        if test_cmd: job['needs'] = ['test']
        jobs[f'deploy-{env_name}'] = job
    wf = {'name': app, 'on': {'push': {'branches': branches}}, 'jobs': jobs}
    import re
    raw = yaml.dump(wf, default_flow_style=False, sort_keys=False, allow_unicode=True)
    raw = re.sub(r"^'on':", 'on:', raw, flags=re.MULTILINE)
    return '# Managed by fastops — regenerate with fo_hook(), do not hand-edit\n' + raw

def fo_hook(token=None, path='.'):
    '''Push .github/workflows/fastops.yml via ghapi, or write locally if no token given.
    Reads .fastops/config.json to build the workflow. Prints required GitHub secrets.'''
    tok = _resolve_gh_token(token)
    cfg = _read_fo_config(path)
    if not cfg: raise ValueError('No .fastops/config.json — run fo_init() first')
    yml = mk_workflow(cfg)
    wf_rel = '.github/workflows/fastops.yml'
    if tok:
        from base64 import b64encode
        owner, repo_name, api = _gh_api(tok, path)
        sha = None
        try: sha = api.repos.get_content(owner, repo_name, wf_rel).sha
        except Exception: pass
        kw = dict(message='chore: update fastops deploy workflow',
                  content=b64encode(yml.encode()).decode())
        if sha: kw['sha'] = sha
        api.repos.create_or_update_file_contents(owner, repo_name, wf_rel, **kw)
        print(f'Workflow pushed to {owner}/{repo_name}')
    else:
        wf = Path(path) / wf_rel
        wf.parent.mkdir(parents=True, exist_ok=True)
        wf.write_text(yml)
        print(f'Workflow written to {wf}')
    _print_required_secrets(cfg)

In [ ]:
#| export
def fo_push_env(envs: dict, token=None, dry_run=False, path='.'):
    '''Push env vars to GitHub via gh CLI. Classifies by env_schema in .fastops/config.json:
    key with None value (or absent from schema) → gh secret set (encrypted).
    key with str value → gh variable set (plaintext).
    dry_run=True: print commands without executing.'''
    cfg = _read_fo_config(path)
    schema = cfg.get('env_schema', {})
    for k, v in envs.items():
        is_secret = schema.get(k, None) is None
        if is_secret:
            cmd = ['gh', 'secret', 'set', k, '--body', str(v)]
            label = f'gh secret set {k} [redacted]'
        else:
            cmd = ['gh', 'variable', 'set', k, '--body', str(v)]
            label = f'gh variable set {k} = {v}'
        if dry_run: print(label)
        else: subprocess.run(cmd, check=True)

_LFS_DEFAULTS = ('*.db', '*.sqlite', '*.tar.gz', '*.onnx', '*.bin')

def fo_lfs(patterns=_LFS_DEFAULTS, path='.'):
    '''Run git lfs install + track for each pattern. Writes .gitattributes — does NOT git add.
    Next: git add .gitattributes && git commit -m "track large files with LFS"'''
    subprocess.run(['git', 'lfs', 'install'], check=True, cwd=str(path))
    for p in patterns:
        subprocess.run(['git', 'lfs', 'track', p], check=True, cwd=str(path))
    print(f'Tracked: {list(patterns)}')
    print('Next: git add .gitattributes && git commit -m "track large files with LFS"')

In [ ]:
#| export
_LOG_DIR = Path.home() / '.fastops' / 'logs'

def _log_error(name, msg, exc=None):
    'Append error to ~/.fastops/logs/<name>.log. POSTs to FASTOPS_ERROR_WEBHOOK if set.'
    _LOG_DIR.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().isoformat()
    entry = f'[{ts}] ERROR {name}: {msg}' + (f'\n  {exc}' if exc else '') + '\n'
    (_LOG_DIR / f'{name}.log').open('a').write(entry)
    if url := os.environ.get('FASTOPS_ERROR_WEBHOOK'):
        import urllib.request as _req
        data = json.dumps({'app': name, 'error': msg, 'time': ts}).encode()
        try: _req.urlopen(_req.Request(url, data=data, headers={'Content-Type': 'application/json'}))
        except Exception: pass

def deploy_safe(env_or_name, *, build=True, health_timeout=60, user='deploy', key=None) -> bool:
    '''Backup → deploy → health_check → auto-rollback on failure. Returns True on success.
    env_or_name: env key in .fastops/config.json (e.g. "prod") or a deploy record name.
    Designed for CI — reads host/config from .fastops/config.json when present.'''
    cfg = _read_fo_config()
    ec = cfg.get('envs', {}).get(env_or_name, {})
    if ec:
        h, p = ec['host'], ec.get('srv_path', '/srv/app')
        name = cfg.get('app', env_or_name)
        ci_cfg = cfg.get('ci', {})
        health_timeout = ci_cfg.get('health_timeout', health_timeout)
        extra_dirs, exclude = ci_cfg.get('extra_dirs'), ci_cfg.get('exclude')
    else:
        h, p = _resolve(env_or_name, '/srv/app')
        name, extra_dirs, exclude = env_or_name, None, None
    if not h: raise ValueError(f'No host found for {env_or_name!r}')

    snap = code_snap(h, p, name, user, key)
    try: backup(name, srv_path=p, user=user, key=key)
    except Exception as e: print(f'Warning: pre-deploy backup failed: {e}')

    def _rb():
        if not snap: return
        try:
            run_ssh(h, f'tar -xzf {snap} -C {p}', user=user, key=key)
            run_ssh(h, f'cd {p} && docker compose up -d --remove-orphans --build', user=user, key=key)
            print(f'Rolled back {name!r} from {snap}')
        except Exception as re: print(f'Rollback failed: {re}')

    try:
        _deploy(Path('.'), h, user=user, key=key, srv_path=p, build=build,
                extra_dirs=extra_dirs, exclude=exclude, snapshot=False)
    except Exception as e:
        _log_error(name, f'deploy failed: {e}', exc=e); _rb(); return False

    try: health_check(h, srv_path=p, timeout=health_timeout, user=user, key=key)
    except Exception as e:
        _log_error(name, f'health check failed: {e}', exc=e); _rb(); return False

    domain = ec.get('domain') or getattr(get_deploy(name) or {}, 'domain', '')
    record_deploy(name, h, domain, Path('.'), srv_path=p)
    return True

In [ ]:
import tempfile as _tf5

with tempfile.TemporaryDirectory() as _td5:
    _tp = Path(_td5)
    subprocess.run(['git', 'init', '-b', 'main', str(_tp)], check=True, capture_output=True)
    (_tp / 'pyproject.toml').write_text('[dev]\ndependencies = ["pytest>=8"]\n')

    fo_init('myapp', '1.2.3.4', 'myapp.example.com', path=_tp)
    cfg = _read_fo_config(_tp)
    assert cfg['app'] == 'myapp'
    assert cfg['envs']['prod']['host'] == '1.2.3.4'
    assert cfg['envs']['prod']['branch'] == 'main'
    assert cfg['ci']['test_cmd'] == 'pytest'
    print('fo_init OK')

    # fo_init with env_schema
    fo_init('myapp', '1.2.3.4', 'myapp.example.com',
            env_schema={'DB_URL': None, 'PORT': '8000'}, path=_tp)
    cfg = _read_fo_config(_tp)
    assert cfg['env_schema'] == {'DB_URL': None, 'PORT': '8000'}
    print('fo_init env_schema OK')

    fo_add_env('staging', '5.6.7.8', 'staging.myapp.com', branch='dev', path=_tp)
    cfg = _read_fo_config(_tp)
    assert 'staging' in cfg['envs']
    wf = _tp / '.github/workflows/fastops.yml'
    assert wf.exists()
    print('fo_add_env OK')

    yml = mk_workflow(cfg)
    assert 'deploy-prod' in yml and 'deploy-staging' in yml
    assert "refs/heads/main" in yml and "refs/heads/dev" in yml
    assert 'uv run pytest' in yml
    assert 'VPS_HOST_PROD' in yml and 'VPS_HOST_STAGING' in yml
    assert '${{ secrets.VPS_SSH_KEY }}' in yml
    assert 'fo_record_deploy' in yml
    # valid YAML
    parsed = yaml.safe_load(yml)
    assert 'jobs' in parsed and parsed.get('name') == 'myapp'
    print('mk_workflow OK')

    fo_record_deploy('prod', 'abc123def456', path=_tp)
    cfg = _read_fo_config(_tp)
    assert cfg['deploys']['prod']['version'] == 'abc123def456'
    fo_status(_tp)
    print('fo_record_deploy + fo_status OK')

    # fo_push_env dry_run
    fo_push_env({'DB_URL': 'postgres://...', 'PORT': '8000'}, dry_run=True, path=_tp)
    print('fo_push_env dry_run OK')

    # teardown_local is an alias
    assert callable(teardown_local)
    print('teardown_local OK')

    # _resolve_gh_token: explicit string always wins
    assert _resolve_gh_token('mytoken') == 'mytoken'
    print('_resolve_gh_token OK')

print('fo_* + mk_workflow tests OK')

## vedicreader example

Deploy [vedicreader](../orgs/vedicreader) — a FastHTML app — to `fastops.angalama.com`.

The app structure in the container:
- `/app/` — built from `vr/` package + `app.py` via uv multistage  
- `/app/data/` — SQLite DB + logs (bind-mount, never in image)  
- `/app/static/` — compiled CSS/JS (bind-mount, rsync'd on deploy)  
- `/app/backups/` — rclone backup target (bind-mount, runtime only)  
- `/app/onnx-community/` — ONNX model files (bind-mount, rsync once)

**Permissions note:** `python_app()` runs as root by default, so usearch can create `.usearch` at runtime without any extra setup. If you add a `USER` instruction, pre-create the dir first with `.run('mkdir -p /app/.usearch')`.

In [ ]:
#| eval: false
src = Path('../orgs/vedicreader')

# Shared deploy config
DOMAIN   = 'fastops.angalama.com'
VOLUMES  = {
    './data':            '/app/data',
    './static':          '/app/static',
    './backups':         '/app/backups',
    './onnx-community':  '/app/onnx-community',
	'./assets':         '/app/assets',
}
# rsync static + models separately; exclude runtime-only dirs from main rsync
SYNC_DIRS = ['static', 'onnx-community', 'assets/']
EXCLUDE   = ['data', 'backups', 'onnx-community', '.venv', '__pycache__', '.git']

# kwargs forwarded to detect_app() → fasthtml_app()
DF_KW = dict(
    pkgs=['ca-certificates', 'rclone', 'libsqlite3-dev'],
    vols=['/app/.config/rclone', '/app/.cache/huggingface'],
    healthcheck='/health',
)

# Inspect generated artefacts without touching any infra
df = detect_app(src, **DF_KW)
print('=== Dockerfile ===')
print(df)

dc = stack(DOMAIN, _port(df), tunnel=True, app_volumes=VOLUMES)
print('\n=== docker-compose.yml ===')
print(dc)

In [ ]:
#| eval: false
keys = read_env_keys(src)
print('Env keys:', keys)

envs = {k: secret_get(k) for k in keys}
check_envs(keys, envs)

In [ ]:
#| eval: false
# Test locally with Multipass first (CF tunnel dials outbound from inside VM)
ip = launch_local(src, DOMAIN, envs=envs, app_volumes=VOLUMES, extra_dirs=SYNC_DIRS, **DF_KW)
print(f'VM IP: {ip}')
print(list_deploys())

In [ ]:
#| eval: false
# Deploy to real Hetzner VPS
ip = launch(src, DOMAIN, envs=envs, app_volumes=VOLUMES, extra_dirs=SYNC_DIRS, exclude=EXCLUDE,**DF_KW)
print(f'Deployed to: {ip}')
print(list_deploys())

In [ ]:
#| eval: false
# Backup and restore
p = backup('vedicreader', data_dirs=['data', 'backups'])
print(f'Backup: {p}')

# restore(p, 'vedicreader')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()